# CS336 Spring 2026 - File-based `train_bpe()` + 多进程 Pre-tokenization
----
本节目标只有一个：
```text
input_path → binary file → safe chunk boundaries
→ local pre-token Counter → Counter aggregation
→ PretokenCounts → BPE core → vocab + ordered merges
```

核心不是“怎么开进程”，而是先保证：

$$
C_{\text{whole}}=\sum_i C_{\text{safe chunk }i}
$$

任意切块并不满足这个等式。

In [ ]:
from __future__ import annotations

from collections import Counter
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path
import os, regex, tempfile

Token = bytes
TokenSequence = tuple[bytes, ...]
PretokenCounts = Counter[TokenSequence]
Pair = tuple[bytes, bytes]

GPT2_PRETOKEN_PATTERN = (
    r"'(?:[sdmt]|ll|ve|re)"
    r"| ?\p{L}+"
    r"| ?\p{N}+"
    r"| ?[^\s\p{L}\p{N}]+"
    r"|\s+(?!\S)"
    r"|\s+"
)


def pretokenize(text):
    return [m.group(0) for m in regex.finditer(GPT2_PRETOKEN_PATTERN, text)]


assert "".join(pretokenize("Hello 世界!")) == "Hello 世界!"
print("pre-tokenization round-trip: 通过")

pre-tokenization round-trip: 通过


## 1. 本节的困难
> 本节真正的困难是: 文件应该在哪切?

假设完整文本为 $T$, 切成
$$
T=T_1+T_2+\cdots+T_k
$$
希望有
$$
C(T)=\sum_{i=1}^k C(T_i)
$$
其中 $C$ 表示 pre-tokenization+frequency counting.
> 但是任意切分文件则会破坏这个条件, 因此后续要解决此问题.


## Q: 为什么不能任意切文件？

A: 完整字符串：

```text
" wonderful"
```

可能是一个 pre-token。

如果切成：

```text
" won" | "derful"
```

再让两个 worker 独立 regex，就变成两个 pre-token。

所以：

> **能成功 UTF-8 decode，不等于分块后的 pre-tokenization 与完整文本等价。**

文件 offset 又是 byte offset，而 UTF-8 一个字符可能占 1～4 bytes，因此 chunking 阶段应明确使用 binary mode：`open(path, "rb")`。

In [3]:
whole = " wonderful"

left = " won"
right = "derful"

print("完整处理", pretokenize(whole))
print("错误处理", pretokenize(left) + pretokenize(right))

assert pretokenize(whole) != pretokenize(left) + pretokenize(right)
print("任意切分确实改变了 segmentation")

完整处理 [' wonderful']
错误处理 [' won', 'derful']
任意切分确实改变了 segmentation


上述还有一个重要的结论:
> 每个 chunk 都能成功 decode, 并不能证明 chunking 是正确的.

## 2. 安全边界：document delimiter

本节利用 `<|endoftext|>`。

训练时 special token 本来就应该：

```text
进入 final vocabulary
但不进入 ordinary BPE pair statistics
```

因此 document A 与 document B 在这个 delimiter 两侧可以独立统计。

算法：

```text
文件大小
→ 先猜近似等距 byte offsets
→ 每个内部 offset 向后搜索 <|endoftext|>
→ 把 boundary 移到 delimiter 起点
→ 去除重复 boundaries
```

如果多个猜测位置最终落到同一个 delimiter，实际 chunk 数少于请求数是正常的：正确性比“强行凑够 worker 数”重要。

In [4]:
def find_chunk_boundaries(file, desired_num_chunks, split_special_token):
    assert isinstance(split_special_token, bytes)
    assert desired_num_chunks >= 1
    assert split_special_token

    file.seek(0, os.SEEK_END)
    file_size = file.tell()
    file.seek(0)

    chunk_size = file_size // desired_num_chunks
    boundaries = [i * chunk_size for i in range(desired_num_chunks + 1)]
    boundaries[-1] = file_size

    read_ahead = 4096

    for i in range(1, len(boundaries) - 1):
        position = boundaries[i]
        file.seek(position)

        while True:
            block = file.read(read_ahead)
            if block == b"":
                boundaries[i] = file_size
                break

            found = block.find(split_special_token)
            if found != -1:
                boundaries[i] = position + found
                break

            position += len(block)

    return sorted(set(boundaries))


## 3. Chunk 内仍要隔离 special tokens

安全 boundary 只决定“在哪里切”。

worker 内部仍然要：

```text
chunk
→ special-token isolation
→ ordinary pieces
→ regex pre-tokenization
→ UTF-8 bytes
→ local Counter
```

而且多个 special token 有重叠时，继续使用“长 token 优先”的规则。